<a href="https://colab.research.google.com/github/diaspj00/Burkholderiales_results/blob/main/Model_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###################################################################################
# BLOCK 1 — Imports + Safe Google Drive Mount (FIXED)
###################################################################################




In [21]:
import os
import csv
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score

# 🔹 SAFE GOOGLE DRIVE MOUNT (idempotente, sem conflitos)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("✔ Google Drive already mounted")

print("✔ Ready to proceed")




✔ Google Drive already mounted
✔ Ready to proceed


###################################################################################
# BLOCK 2 - Seeds, device, paths (Model B)
###################################################################################




In [22]:
SEED_B = 42
MAX_LEN_B = 256
BATCH_SIZE_B = 1   # 🔥 importante para Model B heavy

random.seed(SEED_B)
np.random.seed(SEED_B)
torch.manual_seed(SEED_B)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_B)

# 🔹 determinismo (debug consistente)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE_B = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE_B =", DEVICE_B)
print("BATCH_SIZE_B =", BATCH_SIZE_B)
print("MAX_LEN_B =", MAX_LEN_B)

# 🔹 ROOT comum
ROOT = "/content/drive/MyDrive/contact_maps_project"

# 🔹 dataset
UNIFORM_DIR   = f"{ROOT}/uniform_A"
SPLITS_DIR    = f"{ROOT}/splits_A"

# 🔹 outputs Model B
MODELS_DIR_B  = f"{ROOT}/models_B"
LOGS_DIR_B    = f"{ROOT}/logs_B"
FIGURES_DIR_B = f"{ROOT}/figures_B"

for folder in [MODELS_DIR_B, LOGS_DIR_B, FIGURES_DIR_B]:
    os.makedirs(folder, exist_ok=True)

print("✔ Model B paths initialized")




DEVICE_B = cpu
BATCH_SIZE_B = 1
MAX_LEN_B = 256
✔ Model B paths initialized


###################################################################################
# BLOCK 3 - Amino acid encoding (Model B)
###################################################################################




In [3]:
AA_ALPHABET_B = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX_B = {aa_B: i_B + 1 for i_B, aa_B in enumerate(AA_ALPHABET_B)}

def encode_sequence_B(seq_str_B):
    seq_str_B = seq_str_B.strip().upper()
    return np.array([AA_TO_IDX_B.get(aa_B, 0) for aa_B in seq_str_B], dtype=np.int64)




###################################################################################
# BLOCK 4 - Contact map (Model B)
###################################################################################




In [4]:
def compute_contact_map_B(coords_B, threshold_B=8.0):
    dist_B = np.linalg.norm(coords_B[:, None, :] - coords_B[None, :, :], axis=-1)

    contact_B = (dist_B < threshold_B).astype(np.float32)

    # 🔹 remover contactos triviais i == j
    np.fill_diagonal(contact_B, 0)

    return contact_B




###################################################################################
# BLOCK 5 - Load splits (shared dataset, Model B)
###################################################################################




In [5]:
train_path = f"{SPLITS_DIR}/train_files_A.npy"
val_path   = f"{SPLITS_DIR}/val_files_A.npy"
test_path  = f"{SPLITS_DIR}/test_files_A.npy"

print("Checking split paths...")
print("SPLITS_DIR:", SPLITS_DIR)
print("Available files:",
      os.listdir(SPLITS_DIR) if os.path.exists(SPLITS_DIR) else "Folder missing")

# 🔴 verificação de segurança
if not (
    os.path.exists(train_path) and
    os.path.exists(val_path) and
    os.path.exists(test_path)
):
    raise FileNotFoundError(
        "❌ Splits not found. Run script_prepare_dataset.py first."
    )

# 🔹 carregar splits (partilhados com A1.0 e A2.0)
train_files_B = np.load(train_path, allow_pickle=True)
val_files_B   = np.load(val_path, allow_pickle=True)
test_files_B  = np.load(test_path, allow_pickle=True)

print("✔ train / val / test =",
      len(train_files_B),
      len(val_files_B),
      len(test_files_B))




Checking split paths...
SPLITS_DIR: /content/drive/MyDrive/contact_maps_project/splits_A
Available files: ['train_files_A.npy', 'val_files_A.npy', 'test_files_A.npy']
✔ train / val / test = 843 105 106


###################################################################################
# BLOCK 6 — Dataset (ROBUST VERSION, Model B)
###################################################################################




In [6]:
class ContactDataset_B(Dataset):
    def __init__(self, files_B, max_len_B=256):
        self.files_B = files_B
        self.max_len_B = max_len_B

        # 🔹 pré-filtrar ficheiros válidos
        self.valid_files_B = []
        for f in self.files_B:
            if os.path.exists(f):
                self.valid_files_B.append(f)
            else:
                print(f"⚠️ Missing file skipped at init: {f}")

        if len(self.valid_files_B) == 0:
            raise RuntimeError("❌ No valid files found in dataset.")

        print(f"✔ Dataset B initialized: {len(self.valid_files_B)}/{len(self.files_B)} valid files")

    def __len__(self):
        return len(self.valid_files_B)

    def __getitem__(self, idx_B):

        # 🔁 tentativa robusta (evita crashes)
        for attempt in range(3):

            file_path = self.valid_files_B[idx_B]

            try:
                data_B = np.load(file_path, allow_pickle=True)

                seq_B = data_B["seq"].astype(np.int64)
                contact_B = data_B["contact"].astype(np.float32)

                N_B = int(data_B["length"]) if "length" in data_B else len(seq_B)

                # 🔹 padding
                seq_pad_B = np.zeros(self.max_len_B, dtype=np.int64)
                contact_pad_B = np.zeros((self.max_len_B, self.max_len_B), dtype=np.float32)

                seq_pad_B[:N_B] = seq_B[:N_B]
                contact_pad_B[:N_B, :N_B] = contact_B[:N_B, :N_B]

                return (
                    torch.tensor(seq_pad_B, dtype=torch.long),
                    torch.tensor(contact_pad_B, dtype=torch.float32),
                    torch.tensor(N_B, dtype=torch.long),
                    file_path
                )

            except Exception as e:
                print(f"⚠️ Error loading {file_path}: {e}")

                # 🔹 fallback circular
                idx_B = (idx_B + 1) % len(self.valid_files_B)

        # ❌ fallback extremo
        raise RuntimeError(
            f"❌ Failed to load data after retries starting from index {idx_B}"
        )




###################################################################################
# BLOCK 7 — DataLoaders (Model B, STABLE VERSION FOR COLAB)
###################################################################################




In [7]:
import torch

# 🔹 worker seed control (reprodutibilidade)
def seed_worker(worker_id):
    worker_seed = SEED_B + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# 🔹 generator para controlar shuffle
g_B = torch.Generator()
g_B.manual_seed(SEED_B)

# 🔹 datasets
train_dataset_B = ContactDataset_B(train_files_B, max_len_B=MAX_LEN_B)
val_dataset_B   = ContactDataset_B(val_files_B, max_len_B=MAX_LEN_B)
test_dataset_B  = ContactDataset_B(test_files_B, max_len_B=MAX_LEN_B)

# 🔥 CONFIGURAÇÃO ESTÁVEL PARA COLAB
train_loader_B = DataLoader(
    train_dataset_B,
    batch_size=BATCH_SIZE_B,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=g_B,
    num_workers=0,        # 🔥 essencial no Colab
    pin_memory=False      # 🔥 evita problemas de memória
)

val_loader_B = DataLoader(
    val_dataset_B,
    batch_size=BATCH_SIZE_B,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=g_B,
    num_workers=0,
    pin_memory=False
)

test_loader_B = DataLoader(
    test_dataset_B,
    batch_size=BATCH_SIZE_B,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=g_B,
    num_workers=0,
    pin_memory=False
)

print("✔ DataLoaders B initialized (Colab-safe mode)")




✔ Dataset B initialized: 843/843 valid files
✔ Dataset B initialized: 105/105 valid files
✔ Dataset B initialized: 106/106 valid files
✔ DataLoaders B initialized (Colab-safe mode)


###################################################################################
# BLOCK 8 — Sanity Check (Model B, SMART VERSION)
###################################################################################




In [8]:
import os, json

state_path = f"{LOGS_DIR_B}/training_state_B.json"

# 🔹 controlo manual
FORCE_SANITY_CHECK = False   # força execução
DEBUG_MODE = False           # ativa diagnóstico mesmo em modo skip

# 🔹 flags internas
skip_block8 = False
reason = ""

# 🔹 lógica de decisão
if not FORCE_SANITY_CHECK:
    if os.path.exists(state_path):
        try:
            with open(state_path, "r") as f:
                state = json.load(f)

            status = state.get("status", None)

            if status in ("running", "finished"):
                skip_block8 = True
                reason = f"state={status}"

        except Exception as e:
            skip_block8 = False
            reason = f"state read error: {e}"

# 🔹 execução
if skip_block8 and not DEBUG_MODE:
    print(f"⏭️ BLOCK 8 skipped ({reason})")
else:
    print("▶️ Running BLOCK 8 sanity check...")

    try:
        batch = next(iter(train_loader_B))

        seq_batch, contact_batch, length_batch, source_batch = batch

        print("seq shape =", seq_batch.shape)
        print("contact shape =", contact_batch.shape)
        print("lengths =", length_batch)

        # 🔥 checks adicionais

        # 1. verificar NaNs
        if torch.isnan(seq_batch).any() or torch.isnan(contact_batch).any():
            raise ValueError("❌ NaNs detected in batch")

        # 2. verificar dimensões coerentes
        if seq_batch.shape[1] != MAX_LEN_B:
            raise ValueError(f"❌ Unexpected sequence length: {seq_batch.shape}")

        if contact_batch.shape[1] != MAX_LEN_B:
            raise ValueError(f"❌ Unexpected contact map shape: {contact_batch.shape}")

        # 3. verificar valores válidos
        if contact_batch.max() > 1 or contact_batch.min() < 0:
            raise ValueError("❌ Contact map values out of range [0,1]")

        print("✔ BLOCK 8 sanity check passed")

    except Exception as e:
        print(f"❌ BLOCK 8 failed: {e}")

        if not DEBUG_MODE:
            raise RuntimeError(
                "Sanity check failed. Set DEBUG_MODE=True to inspect without crashing."
            )




▶️ Running BLOCK 8 sanity check...
seq shape = torch.Size([2, 256])
contact shape = torch.Size([2, 256, 256])
lengths = tensor([163, 189])
✔ BLOCK 8 sanity check passed


###################################################################################
# BLOCK 9 - Valid mask (optimized, Model B)
###################################################################################




In [9]:
def build_valid_mask_B(length_B, max_len_B=256, min_sep_B=5):
    idx = np.arange(max_len_B)
    dist = np.abs(idx[:, None] - idx[None, :])
    mask = (dist >= min_sep_B).astype(np.float32)

    # 🔹 cortar ao tamanho real
    mask[length_B:, :] = 0
    mask[:, length_B:] = 0

    return mask




###################################################################################
# BLOCK 10 - Metrics (Model B, compatible)
###################################################################################




In [18]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(y_true_B, y_prob_B, length_B, threshold_B=0.5, max_len=256):

    # 🔹 máscara válida
    mask_B = build_valid_mask_B(length_B, max_len_B=max_len)

    y_true_B = y_true_B[mask_B == 1]
    y_prob_B = y_prob_B[mask_B == 1]

    # 🔹 caso extremo
    if y_true_B.size == 0:
        return np.nan, np.nan

    # 🔹 binarização
    y_pred_B = (y_prob_B > threshold_B).astype(np.int32)

    # 🔹 F1
    if np.sum(y_true_B) == 0:
        f1_B = np.nan
    else:
        f1_B = f1_score(y_true_B, y_pred_B, zero_division=0)

    # 🔹 P@L/5
    L_B = int(length_B)
    k_B = max(1, L_B // 5)
    k_B = min(k_B, y_prob_B.size)

    if k_B <= 0:
        return f1_B, np.nan

    topk_idx_B = np.argsort(y_prob_B)[::-1][:k_B]
    topk_true_B = y_true_B[topk_idx_B]

    pL5_B = float(np.mean(topk_true_B)) if topk_true_B.size > 0 else np.nan

    return f1_B, pL5_B




###################################################################################
# BLOCK 11 - Training helpers (Model B, compatible with A2.0)
###################################################################################




In [19]:
def train_one_epoch_B(model_B, loader_B, optimizer_B, criterion_B, device):
    model_B.train()
    total_loss_B = 0.0

    for seq_B, contact_B, _, _ in loader_B:
        seq_B = seq_B.to(device)
        contact_B = contact_B.to(device)

        optimizer_B.zero_grad()

        logits_B = model_B(seq_B)
        loss_B = criterion_B(logits_B, contact_B)

        loss_B.backward()
        optimizer_B.step()

        total_loss_B += loss_B.item()

    return total_loss_B / len(loader_B)


def validate_one_epoch_B(model_B, loader_B, criterion_B, device, max_len):
    model_B.eval()

    total_loss_B = 0.0
    f1_list_B = []
    pL5_list_B = []

    with torch.no_grad():
        for seq_B, contact_B, length_B, _ in loader_B:

            seq_B = seq_B.to(device)
            contact_B = contact_B.to(device)

            logits_B = model_B(seq_B)
            loss_B = criterion_B(logits_B, contact_B)

            probs_B = torch.sigmoid(logits_B).cpu().numpy()
            targets_B = contact_B.cpu().numpy()
            lengths_B = length_B.cpu().numpy()

            total_loss_B += loss_B.item()

            for i_B in range(seq_B.size(0)):
                f1_B, pL5_B = compute_metrics(
                    targets_B[i_B],
                    probs_B[i_B],
                    int(lengths_B[i_B]),
                    max_len=max_len
                )

                if not np.isnan(f1_B):
                    f1_list_B.append(float(f1_B))
                if not np.isnan(pL5_B):
                    pL5_list_B.append(float(pL5_B))

    avg_loss_B = total_loss_B / len(loader_B) if len(loader_B) > 0 else np.nan
    avg_f1_B = float(np.mean(f1_list_B)) if len(f1_list_B) > 0 else 0.0
    avg_pL5_B = float(np.mean(pL5_list_B)) if len(pL5_list_B) > 0 else 0.0

    return avg_loss_B, avg_f1_B, avg_pL5_B




###################################################################################
# BLOCK 12 — Ranking of structurally informative proteins (Model B)
###################################################################################




In [20]:
import os
import numpy as np
import pandas as pd

def contact_stats_B(npz_path):
    d = np.load(npz_path, allow_pickle=True)

    C = d["contact"].astype(np.float32)
    L = int(d["length"])
    src = str(d["source"]) if "source" in d else os.path.basename(npz_path)

    # 🔹 máscara sem diagonal
    mask = np.ones_like(C, dtype=bool)
    np.fill_diagonal(mask, 0)

    # 🔹 densidade global
    density = C[mask].mean() if mask.sum() > 0 else 0.0

    # 🔹 long-range contacts (|i-j| >= 24)
    idx = np.arange(L)
    ii, jj = np.meshgrid(idx, idx, indexing="ij")
    lr_mask = (np.abs(ii - jj) >= 24)
    lr_mask &= mask

    lr_density = C[lr_mask].mean() if lr_mask.sum() > 0 else 0.0

    # 🔹 entropia (proxy de estrutura)
    p = np.clip(C[mask].mean(), 1e-6, 1-1e-6)
    entropy = -(p*np.log(p) + (1-p)*np.log(1-p))

    return {
        "file": npz_path,
        "protein_id": src,
        "L": L,
        "density": float(density),
        "lr_density": float(lr_density),
        "entropy": float(entropy),
    }

# 🔹 usar dataset comum (partilhado com A1/A2)
files_B = sorted([
    os.path.join(UNIFORM_DIR, f)
    for f in os.listdir(UNIFORM_DIR)
    if f.endswith(".npz")
])

rows_B = [contact_stats_B(f) for f in files_B]
df_stats_B = pd.DataFrame(rows_B)

# 🔹 score combinado (igual ao A2 → fair comparison)
df_stats_B["score"] = (
    0.5 * (df_stats_B["L"] / df_stats_B["L"].max()) +
    0.4 * df_stats_B["lr_density"] +
    0.1 * df_stats_B["density"]
)

# 🔹 ordenar
df_sorted_B = df_stats_B.sort_values("score", ascending=False)

# 🔹 selecionar top-N
TOP_N = 10
df_top_B = df_sorted_B.head(TOP_N).reset_index(drop=True)

print(df_top_B[["protein_id", "L", "density", "lr_density", "score"]])




  protein_id    L   density  lr_density     score
0       1AHI  255  0.040914    0.013920  0.509659
1       1AW1  255  0.040142    0.013174  0.509284
2       1AW2  255  0.039926    0.013062  0.509217
3       1B14  254  0.040553    0.013966  0.507681
4       1B15  254  0.040117    0.013589  0.507487
5       1B16  254  0.040024    0.013439  0.507417
6       1A4U  254  0.039775    0.013439  0.507392
7       1B2L  254  0.039775    0.013363  0.507362
8       1AHH  253  0.041471    0.014164  0.505891
9       1A81  254  0.036756    0.006098  0.504154


In [16]:
"compute_metrics" in globals()

False

###################################################################################
# BLOCK B1 — ResidualBlock_B (final, dilated + residual, A2-consistent)
###################################################################################




In [13]:
import torch
import torch.nn as nn

class ResidualBlock_B(nn.Module):
    def __init__(self, channels_B, dilation_B=1):
        super().__init__()

        self.conv1_B = nn.Conv2d(
            channels_B,
            channels_B,
            kernel_size=3,
            padding=dilation_B,
            dilation=dilation_B
        )
        self.bn1_B = nn.BatchNorm2d(channels_B)

        self.conv2_B = nn.Conv2d(
            channels_B,
            channels_B,
            kernel_size=3,
            padding=dilation_B,
            dilation=dilation_B
        )
        self.bn2_B = nn.BatchNorm2d(channels_B)

        self.relu_B = nn.ReLU()

    def forward(self, x_B):
        identity_B = x_B

        out_B = self.conv1_B(x_B)
        out_B = self.bn1_B(out_B)
        out_B = self.relu_B(out_B)

        out_B = self.conv2_B(out_B)
        out_B = self.bn2_B(out_B)

        # 🔹 residual connection
        out_B = out_B + identity_B
        out_B = self.relu_B(out_B)

        return out_B




###################################################################################
# BLOCK B2 — Model B (Residual + Dilated CNN, A2-consistent)
###################################################################################




In [14]:
import torch
import torch.nn as nn

class ModelB(nn.Module):
    def __init__(self, vocab_size_B=21, embed_dim_B=64, hidden_dim_B=64):
        super().__init__()

        # 🔹 embedding (igual filosofia A2)
        self.embedding_B = nn.Embedding(vocab_size_B, embed_dim_B, padding_idx=0)

        # 🔹 projeção inicial (igual A2 → fair comparison)
        self.input_conv_B = nn.Conv2d(embed_dim_B * 2, hidden_dim_B, kernel_size=1)

        # 🔥 residual + dilated stack (core improvement)
        self.res_blocks_B = nn.Sequential(

            ResidualBlock_B(hidden_dim_B, dilation_B=1),
            ResidualBlock_B(hidden_dim_B, dilation_B=2),
            ResidualBlock_B(hidden_dim_B, dilation_B=4),
            ResidualBlock_B(hidden_dim_B, dilation_B=8),

            # 🔹 opcional: fechar ciclo (melhora estabilidade)
            ResidualBlock_B(hidden_dim_B, dilation_B=4),
            ResidualBlock_B(hidden_dim_B, dilation_B=2),
            ResidualBlock_B(hidden_dim_B, dilation_B=1),
        )

        # 🔹 output layer (igual A2)
        self.output_conv_B = nn.Conv2d(hidden_dim_B, 1, kernel_size=1)

    def forward(self, seq_B):

        # 🔹 embedding
        emb_B = self.embedding_B(seq_B)  # (B, L, D)

        B, L, D = emb_B.shape

        # 🔹 pairwise construction (igual A2)
        emb_i = emb_B.unsqueeze(2).expand(-1, -1, L, -1)
        emb_j = emb_B.unsqueeze(1).expand(-1, L, -1, -1)

        pair_B = torch.cat([emb_i, emb_j], dim=-1)  # (B, L, L, 2D)
        pair_B = pair_B.permute(0, 3, 1, 2)         # (B, 2D, L, L)

        # 🔹 CNN
        x_B = self.input_conv_B(pair_B)
        x_B = self.res_blocks_B(x_B)

        logits_B = self.output_conv_B(x_B).squeeze(1)

        return logits_B




###################################################################################
# BLOCK B3 — Training Model B (robust + resume + anti-rerun + clean retrain)
###################################################################################




In [15]:
import os, csv, json
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(MODELS_DIR_B, exist_ok=True)
os.makedirs(LOGS_DIR_B, exist_ok=True)
os.makedirs(FIGURES_DIR_B, exist_ok=True)

log_csv_path_B = f"{LOGS_DIR_B}/training_log_B.csv"
ckpt_path_B    = f"{MODELS_DIR_B}/checkpoint_B.pt"
best_path_B    = f"{MODELS_DIR_B}/best_model_B.pt"
state_path_B   = f"{LOGS_DIR_B}/training_state_B.json"

# 🔹 controlo manual
FORCE_RETRAIN_B = False

# 🔴 clean retrain
if FORCE_RETRAIN_B:
    print("⚠️ FORCE_RETRAIN_B=True → clearing previous artifacts")

    for path in [log_csv_path_B, ckpt_path_B, best_path_B, state_path_B]:
        if os.path.exists(path):
            os.remove(path)
            print(f"🗑️ Removed: {path}")

# 🔹 proteção contra rerun após fim
if os.path.exists(state_path_B):
    with open(state_path_B, "r") as f:
        state_B = json.load(f)

    if state_B.get("status") == "finished" and not FORCE_RETRAIN_B:
        print("⚠️ Training already completed. Skipping execution.")
        raise RuntimeError("Set FORCE_RETRAIN_B=True to retrain.")

# 🔹 modelo + optimizer + loss
model_B = ModelB().to(DEVICE_B)

optimizer_B = torch.optim.Adam(model_B.parameters(), lr=1e-3)

# 🔥 mesma loss do A2.0 para fair comparison
pos_weight_B = torch.tensor([8.0]).to(DEVICE_B)
criterion_B = nn.BCEWithLogitsLoss(pos_weight=pos_weight_B)

EPOCHS_B   = 30
PATIENCE_B = 5

start_epoch_B = 0
best_val_loss_B = float("inf")
patience_counter_B = 0

# 🔁 resume
if os.path.exists(ckpt_path_B):
    ckpt_B = torch.load(ckpt_path_B, map_location=DEVICE_B)

    model_B.load_state_dict(ckpt_B["model"])
    optimizer_B.load_state_dict(ckpt_B["optimizer"])

    start_epoch_B = ckpt_B["epoch"] + 1
    best_val_loss_B = ckpt_B.get("best_val_loss", float("inf"))
    patience_counter_B = ckpt_B.get("patience", 0)

    print(f"🔁 Resuming Model B from epoch {start_epoch_B}")

# 🔹 criar CSV se não existir
if not os.path.exists(log_csv_path_B):
    with open(log_csv_path_B, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "val_f1", "val_pL5"])

# 🔹 treino
for epoch_B in range(start_epoch_B, EPOCHS_B):

    train_loss_B = train_one_epoch_B(
        model_B,
        train_loader_B,
        optimizer_B,
        criterion_B,
        DEVICE_B
    )

    val_loss_B, val_f1_B, val_pL5_B = validate_one_epoch_B(
        model_B,
        val_loader_B,
        criterion_B,
        DEVICE_B,
        MAX_LEN_B
    )

    print(f"\n[Model B] Epoch {epoch_B+1}/{EPOCHS_B}")
    print(f"Train Loss: {train_loss_B:.4f}")
    print(f"Val Loss:   {val_loss_B:.4f}")
    print(f"Val F1:     {val_f1_B:.4f}")
    print(f"Val P@L5:   {val_pL5_B:.4f}")

    # 🔹 append logs
    with open(log_csv_path_B, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch_B, train_loss_B, val_loss_B, val_f1_B, val_pL5_B])

    # 🔹 best model
    if val_loss_B < best_val_loss_B:
        best_val_loss_B = val_loss_B
        patience_counter_B = 0

        torch.save(model_B.state_dict(), best_path_B)

        print("✅ Best Model B saved")

    else:
        patience_counter_B += 1
        print(f"Patience: {patience_counter_B}/{PATIENCE_B}")

    # 🔹 checkpoint
    torch.save({
        "epoch": epoch_B,
        "model": model_B.state_dict(),
        "optimizer": optimizer_B.state_dict(),
        "best_val_loss": best_val_loss_B,
        "patience": patience_counter_B
    }, ckpt_path_B)

    # 🔹 estado
    with open(state_path_B, "w") as f:
        json.dump({"status": "running", "last_epoch": epoch_B}, f)

    # 🔹 plot incremental
    try:
        df_B = pd.read_csv(log_csv_path_B)

        plt.figure(figsize=(8,5))

        plt.plot(df_B["epoch"], df_B["val_f1"],
                 label="F1", marker='o')

        plt.plot(df_B["epoch"], df_B["val_pL5"],
                 label="P@L5", marker='s')

        plt.plot(df_B["epoch"], df_B["val_loss"],
                 label="Val Loss", linestyle='--')

        plt.xlabel("Epoch")
        plt.ylabel("Metric value")
        plt.title("Model B — Validation Metrics")
        plt.legend()
        plt.grid(True)

        fig_path_B = f"{FIGURES_DIR_B}/model_B_validation_metrics.png"
        plt.savefig(fig_path_B, dpi=300, bbox_inches='tight')
        plt.close()

    except Exception as e:
        print("⚠️ Plot skipped:", e)

    if patience_counter_B >= PATIENCE_B:
        print("⛔ Early stopping Model B")
        break

# 🔹 marcar como finalizado
with open(state_path_B, "w") as f:
    json.dump({"status": "finished", "last_epoch": epoch_B}, f)

print("✔ Model B training finished and state saved")




NameError: name 'compute_metrics' is not defined